In [ ]:
# Cell 2 — Imports
from __future__ import annotations
import os, re, sys, time, math, warnings
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score, matthews_corrcoef, precision_score, recall_score, roc_auc_score, roc_curve, auc
from sklearn.model_selection import KFold, LeaveOneGroupOut
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

In [ ]:
# Cell 3 — Configuration
CT_DIR = Path("/Volumes/My Passport/ct_slices")
ANN_DIR = Path("/Volumes/My Passport/annotations")
CACHE_DIR = Path('Results') / 'Cache'
RESULT_DIR = Path('Results')
FIGURES_DIR = RESULT_DIR / 'Figures'
TABLES_DIR = RESULT_DIR / 'Tables'
MODELS_DIR = RESULT_DIR / 'Models'
PROBABILITY_MAPS_DIR = RESULT_DIR / 'ProbabilityMaps'

VALIDATION_MODE = 'loso'  # 'loso', '5fold', '6fold'
FINAL_TEST_SLICE = None
RANDOM_SEED = 42
TARGET_SLICE_NUMBERS = {151, 350, 500, 700, 800, 900}
MIN_PARTICLE_SIZE = 5
# 'minmax' (per-slice, independent -- safe, no leakage, but scale is inconsistent slice-to-slice)
# 'zscore' (per-slice, independent -- same safety property as minmax)
# 'global_minmax' (a single min/max shared across slices for scale consistency -- but the range
#   MUST be computed only from each fold's training slices, never from a held-out val slice or
#   the whole volume, or it leaks. See fold_intensity_range() in Cell 4 and its use in Cells 9/10/14.)
NORMALIZATION_MODE = 'global_minmax'
NORMALIZATION_EPS = 1e-8
NORMALIZE_INPUTS = True
USE_BACKGROUND_MASK = False  # True: label = particle_mask (1) / background_mask (0) / else ignore (-1)
                              # False: label = particle_mask (1) / everything else (0); background.tif not required
RF_PARAMS = {'n_estimators': 150, 'max_depth': 20, 'min_samples_leaf': 3, 'min_samples_split': 2, 'class_weight': 'balanced', 'n_jobs': -1, 'random_state': RANDOM_SEED}
SAVE_MODELS = True
SAVE_PROBABILITY_MAPS = True
SAVE_ROC = True
SAVE_CONFUSION = True
SAVE_FEATURE_IMPORTANCE = True
SAVE_TABLES = True
SAVE_CACHE = True
FEATURE_NAMES = ['raw_intensity', 'gaussian_sigma_1', 'gaussian_sigma_2', 'gaussian_sigma_3', 'sobel_magnitude', 'log_approx', 'dog', 'local_contrast', 'hessian_lambda_max', 'hessian_lambda_min', 'hessian_frobenius']
EXECUTION_DATE = datetime.now(timezone.utc).isoformat(timespec='seconds')
START_TIME = time.perf_counter()
for directory in [RESULT_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR, PROBABILITY_MAPS_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
np.random.seed(RANDOM_SEED)

In [ ]:
# Cell 4 — Utility functions
import hashlib
import inspect
from scipy.ndimage import label as nd_label

@dataclass(frozen=True)
class SliceInfo:
    slice_number: int
    ct_path: Path
    particle_path: Path
    background_path: Optional[Path] = None

    def annotation_fingerprint(self) -> str:
        """Hash of the actual annotation file bytes, so editing/re-saving a mask on disk
        (same filename, same config) invalidates the cache automatically -- no manual
        cache-clearing required."""
        h = hashlib.sha1()
        h.update(self.particle_path.read_bytes())
        if self.background_path is not None and self.background_path.exists():
            h.update(self.background_path.read_bytes())
        return h.hexdigest()[:10]

    @property
    def cache_path(self) -> Path:
        return CACHE_DIR / f'slice_{self.slice_number:06d}_{compute_config_fingerprint()}_{self.annotation_fingerprint()}.npz'

def seed_everything(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

def load_tiff(path: Path) -> np.ndarray:
    return np.asarray(Image.open(path))

def normalize_image(image: np.ndarray) -> np.ndarray:
    # Used only for the cached per-slice diagnostic fields (SLICE_CACHE[...]['normalized_image'
    # / 'features']) and for the 'minmax'/'zscore' training paths, which are per-slice and safe.
    # When NORMALIZATION_MODE == 'global_minmax', training/evaluation do NOT use this function or
    # its cached output -- see fold_intensity_range() and features_for_slice() below, which apply
    # a fold-specific, train-only range instead.
    image = image.astype(np.float32, copy=False)
    if not NORMALIZE_INPUTS:
        return image
    if NORMALIZATION_MODE == 'zscore':
        return (image - float(image.mean())) / (float(image.std()) + NORMALIZATION_EPS)
    return (image - float(image.min())) / (float(image.max() - image.min()) + NORMALIZATION_EPS)

def fold_intensity_range(train_slice_numbers: Sequence[int]) -> Tuple[float, float]:
    """Min/max computed strictly from a fold's training slices' raw pixel values. Never include
    the held-out validation slice here -- that would leak validation-set information into the
    preprocessing applied to it."""
    raws = [SLICE_CACHE[n]['raw_image'] for n in train_slice_numbers]
    lo = min(float(r.min()) for r in raws)
    hi = max(float(r.max()) for r in raws)
    return lo, hi

def features_for_slice(slice_number: int, intensity_range: Optional[Tuple[float, float]] = None) -> np.ndarray:
    """Fold-aware feature lookup. For 'global_minmax', intensity_range must be the calling fold's
    train-only range (from fold_intensity_range), applied identically whether slice_number is one
    of that fold's training slices or its held-out validation slice -- this is the standard
    fit(train).transform(test) pattern, not leakage, because the range's *parameters* only ever
    come from training data."""
    if NORMALIZATION_MODE == 'global_minmax':
        if intensity_range is None:
            raise ValueError('global_minmax requires a fold-specific (train-only) intensity_range')
        lo, hi = intensity_range
        raw = SLICE_CACHE[slice_number]['raw_image'].astype(np.float32, copy=False)
        norm = (raw - lo) / (hi - lo + NORMALIZATION_EPS)
        return extract_features(norm)
    return SLICE_CACHE[slice_number]['features']

def remove_small_blobs(mask: np.ndarray, min_size: int) -> np.ndarray:
    labeled, _ = nd_label(mask)
    counts = np.bincount(labeled.ravel())
    cleaned = mask.copy()
    for label_id, size in enumerate(counts):
        if label_id != 0 and size < min_size:
            cleaned[labeled == label_id] = False
    return cleaned

def load_labels(particle_path: Path, background_path: Optional[Path]) -> Tuple[np.ndarray, np.ndarray]:
    """Returns (train_labels, eval_labels).

    train_labels: what the model is trained on. If USE_BACKGROUND_MASK, this is the sparse,
    curated labeling (-1 for anything the annotator didn't explicitly mark, 0 for clicked
    background, 1 for particle) -- i.e. training only sees confidently-annotated pixels.
    Otherwise it's the same closed-world labeling as eval_labels (no curation to decouple).

    eval_labels: what every configuration is SCORED against, always. Closed-world, full-image:
    particle_mask -> 1, everything else -> 0, no -1 anywhere. This is what
    USE_BACKGROUND_MASK=False already produced for training; now it's used for evaluation
    unconditionally, so results are comparable across both label-curation strategies."""
    particle = load_tiff(particle_path)
    if particle.dtype.kind == 'f':
        particle_mask = particle > 0.001
    else:
        particle_mask = particle > 127 if particle.max() > 1 else particle == 1
    particle_mask = remove_small_blobs(particle_mask, MIN_PARTICLE_SIZE)

    eval_labels = np.zeros(particle_mask.shape, dtype=np.int8)
    eval_labels[particle_mask] = 1

    if not USE_BACKGROUND_MASK:
        return eval_labels.copy(), eval_labels

    background = load_tiff(background_path)
    if background.dtype.kind == 'f':
        background_mask = background > 0.001
    else:
        background_mask = background > 127 if background.max() > 1 else background == 1
    train_labels = np.full(particle_mask.shape, -1, dtype=np.int8)
    train_labels[background_mask] = 0
    train_labels[particle_mask] = 1
    return train_labels, eval_labels

def compute_config_fingerprint() -> str:
    """Hashes everything that affects cached slice contents (config values + the actual
    labeling/feature-extraction code), so changing any of them invalidates old cache files
    instead of silently reusing stale features. Annotation file content is handled separately
    by SliceInfo.annotation_fingerprint(), since it's per-slice, not global config."""
    payload = '|'.join([
        repr(MIN_PARTICLE_SIZE), repr(NORMALIZATION_MODE), repr(NORMALIZATION_EPS),
        repr(NORMALIZE_INPUTS), repr(USE_BACKGROUND_MASK), repr(tuple(FEATURE_NAMES)),
        inspect.getsource(load_labels), inspect.getsource(extract_features),
    ])
    return hashlib.sha1(payload.encode()).hexdigest()[:10]

def save_figure(fig: plt.Figure, path: Path) -> None:
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)

def summary_line(name: str, value: Any) -> None:
    print(f'{name}: {value}')

def slice_number_from_name(name: str) -> Optional[int]:
    match = re.search(r'annotation_slice_(\d+)_particle\.tiff?$', name)
    return None if match is None else int(match.group(1))

In [ ]:
# Cell 5 — Automatic dataset discovery
def discover_dataset() -> Tuple[List[SliceInfo], Dict[str, Any]]:
    particle_files = sorted(
        list(ANN_DIR.glob('annotation_slice_*_particle.tif')) +
        list(ANN_DIR.glob('annotation_slice_*_particle.tiff'))
    )
    slice_infos: List[SliceInfo] = []
    missing_files, duplicates, ignored_files = [], [], []
    seen: Dict[int, int] = {}

    def resolve_annotation_file(slice_num: int, kind: str) -> Optional[Path]:
        candidates = [
            ANN_DIR / f'annotation_slice_{slice_num:04d}_{kind}.tif',
            ANN_DIR / f'annotation_slice_{slice_num:04d}_{kind}.tiff',
        ]
        for candidate in candidates:
            if candidate.exists():
                return candidate
        return None

    def resolve_ct_file(slice_num: int) -> Path:
        # Files on this drive are prefixed with "Copy of " (e.g. "Copy of slice_000350.tif").
        # Fall back to the unprefixed name in case that's ever fixed upstream.
        candidates = [
            CT_DIR / f'Copy of slice_{slice_num:06d}.tif',
            CT_DIR / f'slice_{slice_num:06d}.tif',
        ]
        for candidate in candidates:
            if candidate.exists():
                return candidate
        return candidates[-1]

    for particle_path in particle_files:
        slice_num = slice_number_from_name(particle_path.name)
        if slice_num is None:
            ignored_files.append(particle_path.name)
            continue
        if slice_num not in TARGET_SLICE_NUMBERS:
            ignored_files.append(particle_path.name)
            continue
        seen[slice_num] = seen.get(slice_num, 0) + 1
        background_path = resolve_annotation_file(slice_num, 'background')
        ct_path = resolve_ct_file(slice_num)
        if not ct_path.exists():
            missing_files.append(ct_path.name)
            continue
        if background_path is None and USE_BACKGROUND_MASK:
            missing_files.append(f'annotation_slice_{slice_num:04d}_background.tif/.tiff')
        if background_path is not None or not USE_BACKGROUND_MASK:
            slice_infos.append(SliceInfo(slice_num, ct_path, particle_path, background_path))

    duplicates = sorted([s for s, count in seen.items() if count > 1])
    report = {
        'n_particle_files': len(particle_files),
        'n_valid_slices': len(slice_infos),
        'missing_files': sorted(set(missing_files)),
        'duplicates': duplicates,
        'ignored_files': sorted(set(ignored_files)),
        'target_slice_numbers': sorted(TARGET_SLICE_NUMBERS),
    }
    return sorted(slice_infos, key=lambda x: x.slice_number), report

SLICE_INFOS, DISCOVERY_REPORT = discover_dataset()
seed_everything(RANDOM_SEED)
print('Dataset summary')
summary_line('CT_DIR', CT_DIR)
summary_line('ANN_DIR', ANN_DIR)
summary_line('Target slices', sorted(TARGET_SLICE_NUMBERS))
summary_line('Using background mask', USE_BACKGROUND_MASK)
summary_line('Valid slices', len(SLICE_INFOS))
summary_line('Missing files', len(DISCOVERY_REPORT['missing_files']))
summary_line('Duplicates', len(DISCOVERY_REPORT['duplicates']))
summary_line('Ignored files', len(DISCOVERY_REPORT['ignored_files']))
if DISCOVERY_REPORT['missing_files']:
    print('Missing files:', DISCOVERY_REPORT['missing_files'])
if DISCOVERY_REPORT['duplicates']:
    print('Duplicates:', DISCOVERY_REPORT['duplicates'])
if DISCOVERY_REPORT['ignored_files']:
    print('Ignored files:', DISCOVERY_REPORT['ignored_files'])

In [ ]:
# Cell 6 — Feature extraction
from scipy.ndimage import gaussian_filter, uniform_filter
from skimage.feature import hessian_matrix, hessian_matrix_eigvals
from skimage.filters import sobel

def extract_features(image: np.ndarray) -> np.ndarray:
    image = image.astype(np.float32, copy=False)
    g1, g2, g3 = gaussian_filter(image, 1), gaussian_filter(image, 2), gaussian_filter(image, 3)
    sobel_mag = np.sqrt(sobel(image, axis=1) ** 2 + sobel(image, axis=0) ** 2)
    local_contrast = image - uniform_filter(image, size=5)
    h_elems = hessian_matrix(image, sigma=2, order='rc')
    eig = hessian_matrix_eigvals(h_elems)
    stack = np.stack([image, g1, g2, g3, sobel_mag, g1 - g3, g1 - g2, local_contrast, eig[0], eig[1], np.sqrt(eig[0] ** 2 + eig[1] ** 2)], axis=-1)
    return stack.reshape(-1, 11)

def prepare_slice(slice_info: SliceInfo) -> Dict[str, Any]:
    if not slice_info.ct_path.exists():
        raise FileNotFoundError(f'CT slice not found: {slice_info.ct_path}')
    image_path = slice_info.ct_path
    image = normalize_image(load_tiff(image_path))
    train_labels, eval_labels = load_labels(slice_info.particle_path, slice_info.background_path)
    return {'slice_number': slice_info.slice_number, 'raw_image': load_tiff(image_path), 'normalized_image': image, 'train_labels': train_labels, 'eval_labels': eval_labels, 'features': extract_features(image)}

FEATURE_NAMES = FEATURE_NAMES
print('Feature matrix shape for a dummy 8x8 slice:', extract_features(np.zeros((8, 8), dtype=np.float32)).shape)

In [ ]:
# Cell 7 — Cache system
def cache_slice(slice_info: SliceInfo) -> Dict[str, Any]:
    if slice_info.cache_path.exists():
        with np.load(slice_info.cache_path, allow_pickle=True) as cached:
            return {'slice_number': int(cached['slice_number'][0]), 'raw_image': cached['raw_image'], 'normalized_image': cached['normalized_image'], 'train_labels': cached['train_labels'], 'eval_labels': cached['eval_labels'], 'features': cached['features']}
    data = prepare_slice(slice_info)
    if SAVE_CACHE:
        np.savez_compressed(slice_info.cache_path, slice_number=np.array([data['slice_number']], dtype=np.int32), raw_image=data['raw_image'], normalized_image=data['normalized_image'], train_labels=data['train_labels'], eval_labels=data['eval_labels'], features=data['features'])
    return data

SLICE_CACHE: Dict[int, Dict[str, Any]] = {}
for info in tqdm(SLICE_INFOS, desc='Caching / loading slices'):
    SLICE_CACHE[info.slice_number] = cache_slice(info)
print(f'Prepared {len(SLICE_CACHE)} slices')

In [ ]:
# Cell 7b — Label coverage diagnostic
# This shows TRAINING coverage only (train_labels). Evaluation is now always full-image
# (eval_labels has no -1), so there's nothing to report there -- every fold's eval always
# covers 100% / n_pixels = image size, by construction.
coverage_rows = []
for n, data in sorted(SLICE_CACHE.items()):
    labels = data['train_labels']
    total = labels.size
    n_bg = int((labels == 0).sum())
    n_particle = int((labels == 1).sum())
    n_unlabeled = int((labels == -1).sum())
    coverage_rows.append({
        'slice_number': n,
        'total_pixels': total,
        'pct_background': round(100 * n_bg / total, 2),
        'pct_particle': round(100 * n_particle / total, 2),
        'pct_unlabeled_excluded_from_training': round(100 * n_unlabeled / total, 2),
    })
coverage_df = pd.DataFrame(coverage_rows)
print('Training-label coverage per slice (unlabeled pixels are excluded from training only; evaluation is always full-image):')
print(coverage_df.to_string(index=False))
if SAVE_TABLES:
    coverage_df.to_csv(TABLES_DIR / 'label_coverage.csv', index=False)

In [ ]:
# Cell 8 — Cross-validation fold generation
def create_folds(slice_infos: Sequence[SliceInfo], mode: str, final_test_slice: Optional[int] = None) -> Tuple[List[Dict[str, List[int]]], List[SliceInfo], Optional[SliceInfo]]:
    ordered = sorted(slice_infos, key=lambda s: s.slice_number)
    holdout = next((s for s in ordered if s.slice_number == final_test_slice), None) if final_test_slice is not None else None
    cv_slices = [s for s in ordered if s.slice_number != final_test_slice] if holdout is not None else ordered
    if not cv_slices: raise ValueError('No slices available for cross-validation')
    numbers = np.array([s.slice_number for s in cv_slices])
    folds = []
    if mode == 'loso':
        splitter = LeaveOneGroupOut()
        for i, (tr, va) in enumerate(splitter.split(numbers, groups=numbers), start=1):
            folds.append({'fold_id': i, 'train_slices': numbers[tr].tolist(), 'val_slices': numbers[va].tolist()})
    else:
        k = 5 if mode == '5fold' else 6
        k = min(k, len(cv_slices))
        splitter = KFold(n_splits=max(2, k), shuffle=True, random_state=RANDOM_SEED)
        for i, (tr, va) in enumerate(splitter.split(numbers), start=1):
            folds.append({'fold_id': i, 'train_slices': numbers[tr].tolist(), 'val_slices': numbers[va].tolist()})
    return folds, cv_slices, holdout

FOLDS, CV_SLICE_INFOS, HOLDOUT_SLICE_INFO = create_folds(SLICE_INFOS, VALIDATION_MODE, FINAL_TEST_SLICE)
print(f'Validation mode: {VALIDATION_MODE}; folds: {len(FOLDS)}; holdout: {None if HOLDOUT_SLICE_INFO is None else HOLDOUT_SLICE_INFO.slice_number}')

# Reproducibility manifest: exactly which slices (and their source files) are train vs val
# in each fold. All slices here come from a single specimen/scan (no specimen-level grouping
# exists in this dataset), so this records the finest grouping unit that's actually available.
import json
_path_by_slice = {s.slice_number: s for s in CV_SLICE_INFOS}
fold_manifest = {
    'validation_mode': VALIDATION_MODE,
    'holdout_slice': None if HOLDOUT_SLICE_INFO is None else HOLDOUT_SLICE_INFO.slice_number,
    'folds': [
        {
            'fold_id': fold['fold_id'],
            'train_slices': fold['train_slices'],
            'val_slices': fold['val_slices'],
            'train_files': [_path_by_slice[n].ct_path.name for n in fold['train_slices']],
            'val_files': [_path_by_slice[n].ct_path.name for n in fold['val_slices']],
        }
        for fold in FOLDS
    ],
}
if SAVE_TABLES:
    with open(TABLES_DIR / 'fold_manifest.json', 'w') as f:
        json.dump(fold_manifest, f, indent=2)
print(f"Fold manifest written to {TABLES_DIR / 'fold_manifest.json'}")

In [27]:
# Cell 8 — Debug listing for annotation files
from pathlib import Path

ann_dir = Path("/Volumes/My Passport/annotations/annotations")
print("Particle tif:", [p.name for p in ann_dir.glob("annotation_slice_*_particle.tif") if slice_number_from_name(p.name) in TARGET_SLICE_NUMBERS])
print("Particle tiff:", [p.name for p in ann_dir.glob("annotation_slice_*_particle.tiff") if slice_number_from_name(p.name) in TARGET_SLICE_NUMBERS])
print("Background tif:", [p.name for p in ann_dir.glob("annotation_slice_*_background.tif") if slice_number_from_name(p.name) in TARGET_SLICE_NUMBERS])
print("Background tiff:", [p.name for p in ann_dir.glob("annotation_slice_*_background.tiff") if slice_number_from_name(p.name) in TARGET_SLICE_NUMBERS])

Particle tif: ['annotation_slice_0151_particle.tif', 'annotation_slice_0350_particle.tif']
Particle tiff: []
Background tif: []
Background tiff: []


In [ ]:
# Cell 9 — Training
def assemble_matrix(slice_numbers: Sequence[int], intensity_range: Optional[Tuple[float, float]] = None) -> Tuple[np.ndarray, np.ndarray]:
    xs, ys = [], []
    for n in slice_numbers:
        labels = SLICE_CACHE[n]['train_labels'].ravel()
        mask = labels != -1
        xs.append(features_for_slice(n, intensity_range)[mask])
        ys.append(labels[mask])
    return np.vstack(xs), np.concatenate(ys)

def train_model(x: np.ndarray, y: np.ndarray) -> RandomForestClassifier:
    model = RandomForestClassifier(**RF_PARAMS)
    model.fit(x, y)
    return model

trained_models: Dict[int, RandomForestClassifier] = {}
FOLD_INTENSITY_RANGES: Dict[int, Optional[Tuple[float, float]]] = {}
TRAIN_START = time.perf_counter()
for fold in tqdm(FOLDS, desc='Training folds'):
    fold_id = fold['fold_id']
    # Computed strictly from this fold's training slices -- never the held-out val slice.
    intensity_range = fold_intensity_range(fold['train_slices']) if NORMALIZATION_MODE == 'global_minmax' else None
    FOLD_INTENSITY_RANGES[fold_id] = intensity_range
    x_train, y_train = assemble_matrix(fold['train_slices'], intensity_range)
    trained_models[fold_id] = train_model(x_train, y_train)
    if SAVE_MODELS:
        joblib.dump(trained_models[fold_id], MODELS_DIR / f'rf_fold_{fold_id:02d}.joblib')
FINAL_MODEL = None
FINAL_INTENSITY_RANGE = None
if HOLDOUT_SLICE_INFO is not None:
    final_train_slices = [s.slice_number for s in CV_SLICE_INFOS]
    FINAL_INTENSITY_RANGE = fold_intensity_range(final_train_slices) if NORMALIZATION_MODE == 'global_minmax' else None
    x_all, y_all = assemble_matrix(final_train_slices, FINAL_INTENSITY_RANGE)
    FINAL_MODEL = train_model(x_all, y_all)
    if SAVE_MODELS:
        joblib.dump(FINAL_MODEL, MODELS_DIR / 'rf_final_model.joblib')
TRAIN_END = time.perf_counter()
print(f'Training time: {TRAIN_END - TRAIN_START:.2f} sec')

In [ ]:
# Cell 10 — Evaluation
def eval_fold(model: RandomForestClassifier, slice_numbers: Sequence[int], fold_id: int, intensity_range: Optional[Tuple[float, float]] = None) -> Dict[str, Any]:
    records, y_true_all, y_pred_all, y_prob_all = [], [], [], []
    for n in slice_numbers:
        # Always score against eval_labels: closed-world, full-image, no -1. This is what makes
        # USE_BACKGROUND_MASK=True and =False directly comparable -- the *training* labels differ
        # (curated vs full-image), but the evaluation population is identical in both cases.
        labels = SLICE_CACHE[n]['eval_labels'].ravel(); mask = labels != -1
        x = features_for_slice(n, intensity_range)[mask]; y_true = labels[mask]
        y_prob = model.predict_proba(x)[:, 1]; y_pred = (y_prob >= 0.5).astype(np.int8)
        records.append({'fold_id': fold_id, 'slice_number': n, 'y_true': y_true, 'y_pred': y_pred, 'y_prob': y_prob})
        y_true_all.append(y_true); y_pred_all.append(y_pred); y_prob_all.append(y_prob)
    return {'fold_id': fold_id, 'records': records, 'y_true': np.concatenate(y_true_all), 'y_pred': np.concatenate(y_pred_all), 'y_prob': np.concatenate(y_prob_all)}

EVALUATION_RESULTS = []
EVAL_START = time.perf_counter()
for fold in tqdm(FOLDS, desc='Evaluating folds'):
    EVALUATION_RESULTS.append(eval_fold(trained_models[fold['fold_id']], fold['val_slices'], fold['fold_id'], FOLD_INTENSITY_RANGES[fold['fold_id']]))
HOLDOUT_RESULT = eval_fold(FINAL_MODEL, [HOLDOUT_SLICE_INFO.slice_number], 0, FINAL_INTENSITY_RANGE) if (HOLDOUT_SLICE_INFO is not None and FINAL_MODEL is not None) else None
EVAL_END = time.perf_counter()
print(f'Evaluation time: {EVAL_END - EVAL_START:.2f} sec')

In [ ]:
# Cell 11 — Metrics
# Primary metrics for this imbalanced pixel-level task: ROC AUC, PR AUC (average precision),
# F1, MCC. Accuracy/precision/recall/specificity are kept as secondary/supplementary stats —
# raw accuracy is misleading here given the background/particle pixel imbalance.
PRIMARY_METRICS = ['roc_auc', 'pr_auc', 'f1', 'mcc']
SECONDARY_METRICS = ['accuracy', 'precision', 'recall', 'specificity', 'balanced_accuracy']

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray) -> Dict[str, Any]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1]); tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    sensitivity = recall_score(y_true, y_pred, zero_division=0)
    try: roc_val = roc_auc_score(y_true, y_prob)
    except ValueError: roc_val = np.nan
    try: pr_auc_val = average_precision_score(y_true, y_prob)
    except ValueError: pr_auc_val = np.nan
    return {'roc_auc': roc_val, 'pr_auc': pr_auc_val, 'f1': f1_score(y_true, y_pred, zero_division=0), 'mcc': matthews_corrcoef(y_true, y_pred),
            'accuracy': accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0), 'recall': sensitivity, 'specificity': specificity, 'sensitivity': sensitivity, 'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
            'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp), 'n_pixels': int(len(y_true)), 'confusion_matrix': cm}

metric_rows, confusion_rows = [], []
for result in EVALUATION_RESULTS:
    m = compute_metrics(result['y_true'], result['y_pred'], result['y_prob'])
    metric_rows.append({'fold_id': result['fold_id'], **{k: v for k, v in m.items() if k != 'confusion_matrix'}})
    confusion_rows.append({'fold_id': result['fold_id'], 'tn': m['tn'], 'fp': m['fp'], 'fn': m['fn'], 'tp': m['tp']})
if HOLDOUT_RESULT is not None:
    m = compute_metrics(HOLDOUT_RESULT['y_true'], HOLDOUT_RESULT['y_pred'], HOLDOUT_RESULT['y_prob'])
    metric_rows.append({'fold_id': 'final_test', **{k: v for k, v in m.items() if k != 'confusion_matrix'}})
    confusion_rows.append({'fold_id': 'final_test', 'tn': m['tn'], 'fp': m['fp'], 'fn': m['fn'], 'tp': m['tp']})

metrics_per_fold_df = pd.DataFrame(metric_rows)
metrics_per_fold_df = metrics_per_fold_df[['fold_id'] + PRIMARY_METRICS + SECONDARY_METRICS + ['tn', 'fp', 'fn', 'tp', 'n_pixels']]

# n=6 folds is too small for a meaningful normal-approximation or bootstrap CI (the previous
# mean +/- 1.96*std/sqrt(n) approach could and did produce values outside the valid range for
# bounded metrics). Report median with [min, max] instead -- these are always real observed
# values, so they can never fall outside a metric's valid range.
numeric_cols = metrics_per_fold_df.select_dtypes(include=[np.number]).drop(columns=['fold_id'], errors='ignore')
summary_statistics_df = pd.DataFrame({
    'median': numeric_cols.median(),
    'min': numeric_cols.min(),
    'max': numeric_cols.max(),
})
combined_confusion_matrix = np.sum([confusion_matrix(r['y_true'], r['y_pred'], labels=[0, 1]) for r in EVALUATION_RESULTS], axis=0)
print('Per-fold metrics (primary: roc_auc, pr_auc, f1, mcc | secondary: accuracy, precision, recall, specificity, balanced_accuracy)')
print(metrics_per_fold_df)
print('\nSummary (median [min, max] across folds -- n=6 is too small for a normal-approx or bootstrap CI)')
print(summary_statistics_df)

In [21]:
# Cell 12 — ROC analysis
mean_fpr = np.linspace(0, 1, 1001)
tprs, aucs = [], []
fig, ax = plt.subplots(figsize=(8, 6))
for result in EVALUATION_RESULTS:
    fpr, tpr, _ = roc_curve(result['y_true'], result['y_prob'])
    auc_value = auc(fpr, tpr)
    aucs.append(auc_value)
    fold_id = result['fold_id']
    ax.plot(fpr, tpr, lw=1, alpha=0.3, label=f'Fold {fold_id} AUC={auc_value:.3f}')
    interp = np.interp(mean_fpr, fpr, tpr)
    interp[0] = 0.0
    tprs.append(interp)
mean_tpr = np.mean(tprs, axis=0)
std_tpr = np.std(tprs, axis=0)
mean_tpr[-1] = 1.0
ax.plot(mean_fpr, mean_tpr, color='black', lw=2.5, label=f'Mean ROC AUC={auc(mean_fpr, mean_tpr):.3f}')
ax.fill_between(mean_fpr, np.maximum(mean_tpr - std_tpr, 0), np.minimum(mean_tpr + std_tpr, 1), color='gray', alpha=0.2, label='±1 SD')
ax.plot([0, 1], [0, 1], '--', color='red', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Analysis')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
fig.tight_layout()
if SAVE_ROC:
    save_figure(fig, FIGURES_DIR / 'roc_analysis.png')
else:
    plt.close(fig)

In [22]:
def plot_cm(ax, cm, title, normalize=False):
    m = cm.astype(float)
    if normalize:
        s = m.sum(axis=1, keepdims=True); s[s == 0] = 1; m = m / s
    ax.imshow(m, cmap='Blues')
    for i in range(2):
        for j in range(2):
            txt = f'{m[i,j]:.2f}' if normalize else f'{int(m[i,j])}'
            ax.text(j, i, txt, ha='center', va='center', color='white' if m[i, j] > m.max() / 2 else 'black')
    ax.set_xticks([0,1]); ax.set_yticks([0,1]); ax.set_xticklabels(['Background','Particle']); ax.set_yticklabels(['Background','Particle']); ax.set_title(title)

ind_cms = [confusion_matrix(r['y_true'], r['y_pred'], labels=[0,1]) for r in EVALUATION_RESULTS]
cols = min(3, len(ind_cms)) or 1; rows = int(math.ceil(len(ind_cms) / cols)) or 1
fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4.5*rows))
axes = np.array(axes).reshape(-1)
for ax, cm, res in zip(axes, ind_cms, EVALUATION_RESULTS): plot_cm(ax, cm, f"Fold {res['fold_id']}")
for ax in axes[len(ind_cms):]: ax.axis('off')
fig.tight_layout();
if SAVE_CONFUSION: save_figure(fig, FIGURES_DIR / 'individual_confusion_matrices.png')
else: plt.close(fig)
fig, axes = plt.subplots(1,2, figsize=(10,4.5)); plot_cm(axes[0], combined_confusion_matrix, 'Combined'); plot_cm(axes[1], combined_confusion_matrix, 'Normalized', True); fig.tight_layout()
if SAVE_CONFUSION: save_figure(fig, FIGURES_DIR / 'combined_confusion_matrices.png')
else: plt.close(fig)

In [ ]:
# Cell 14 — Feature importance
impurity_rows, perm_rows = [], []
for result in EVALUATION_RESULTS:
    model = trained_models[result['fold_id']]
    fold_id = result['fold_id']
    intensity_range = FOLD_INTENSITY_RANGES[fold_id]
    impurity_rows.append(pd.Series(model.feature_importances_, index=FEATURE_NAMES, name=f'Fold {fold_id}'))
    val_data = list(result['records'])
    # eval_labels never has -1, so no mask is needed here -- matches how eval_fold built y_true.
    X_val = np.vstack([features_for_slice(r['slice_number'], intensity_range) for r in val_data])
    y_val = np.concatenate([r['y_true'] for r in val_data])
    perm = permutation_importance(model, X_val, y_val, n_repeats=10, random_state=RANDOM_SEED, n_jobs=-1, scoring='f1')
    perm_rows.append(pd.Series(perm.importances_mean, index=FEATURE_NAMES, name=f'Fold {fold_id}'))
impurity_df = pd.DataFrame(impurity_rows)
permutation_df = pd.DataFrame(perm_rows)
feature_importance_df = pd.DataFrame({'feature': FEATURE_NAMES, 'impurity_mean': impurity_df.mean().values, 'impurity_std': impurity_df.std().values, 'permutation_mean': permutation_df.mean().values, 'permutation_std': permutation_df.std().values}).sort_values('impurity_mean', ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feature_importance_df['feature'], feature_importance_df['impurity_mean'], xerr=feature_importance_df['impurity_std'], color='steelblue', alpha=0.85, label='Impurity')
ax.barh(feature_importance_df['feature'], feature_importance_df['permutation_mean'], xerr=feature_importance_df['permutation_std'], color='darkorange', alpha=0.55, label='Permutation')
ax.set_xlabel('Importance')
ax.set_title('Feature Importance')
ax.legend()
ax.grid(axis='x', alpha=0.2)
fig.tight_layout()
if SAVE_FEATURE_IMPORTANCE:
    save_figure(fig, FIGURES_DIR / 'feature_importance.png')
else:
    plt.close(fig)

In [ ]:
# Cell 15 — Probability maps
def save_probability_map(slice_info: SliceInfo, record: Dict[str, Any], path: Path) -> None:
    image = load_tiff(slice_info.ct_path)
    train_labels = SLICE_CACHE[slice_info.slice_number]['train_labels']
    eval_labels = SLICE_CACHE[slice_info.slice_number]['eval_labels']
    prob_map = np.zeros_like(eval_labels, dtype=np.float32)
    mask = eval_labels.ravel() != -1
    prob_map.ravel()[mask] = record['y_prob']
    pred_mask = np.full_like(eval_labels, -1, dtype=np.int8)
    pred_mask.ravel()[mask] = record['y_pred']
    fig, axes = plt.subplots(1, 5, figsize=(20, 4.5))
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('Original CT')
    axes[0].axis('off')
    axes[1].imshow(image, cmap='gray')
    # Shows curated training labels (with -1/unlabeled masked out), not eval_labels, so you can
    # see what the model was actually trained on vs. the full-image ground truth it's scored against.
    axes[1].imshow(np.ma.masked_equal(train_labels, -1), cmap='coolwarm', alpha=0.45, vmin=0, vmax=1)
    axes[1].set_title('Training Labels (curated)')
    axes[1].axis('off')
    im = axes[2].imshow(prob_map, cmap='viridis', vmin=0, vmax=1)
    axes[2].set_title('Probability Map')
    axes[2].axis('off')
    fig.colorbar(im, ax=axes[2], fraction=0.046)
    axes[3].imshow(pred_mask, cmap='coolwarm', vmin=0, vmax=1)
    axes[3].set_title('Predicted Mask')
    axes[3].axis('off')
    axes[4].imshow(image, cmap='gray')
    axes[4].imshow(np.ma.masked_where(pred_mask != 1, pred_mask), cmap='autumn', alpha=0.5, vmin=0, vmax=1)
    axes[4].set_title('Overlay')
    axes[4].axis('off')
    fig.suptitle(f'Slice {slice_info.slice_number:06d}')
    fig.tight_layout()
    save_figure(fig, path)

if SAVE_PROBABILITY_MAPS:
    for fold in EVALUATION_RESULTS:
        fold_id = fold['fold_id']
        for record in fold['records']:
            info = next(s for s in SLICE_INFOS if s.slice_number == record['slice_number'])
            save_probability_map(info, record, PROBABILITY_MAPS_DIR / f'slice_{info.slice_number:06d}_fold_{fold_id}.png')
    if HOLDOUT_RESULT is not None and HOLDOUT_SLICE_INFO is not None:
        save_probability_map(HOLDOUT_SLICE_INFO, HOLDOUT_RESULT['records'][0], PROBABILITY_MAPS_DIR / f'slice_{HOLDOUT_SLICE_INFO.slice_number:06d}_final_test.png')

In [25]:
# Cell 16 — CSV export
if SAVE_TABLES:
    metrics_per_fold_df.to_csv(TABLES_DIR / 'metrics_per_fold.csv', index=False)
    summary_statistics_df.to_csv(TABLES_DIR / 'summary_statistics.csv')
    feature_importance_df.to_csv(TABLES_DIR / 'feature_importance.csv', index=False)
    permutation_df.mean(axis=0).rename('permutation_mean').reset_index().rename(columns={'index': 'feature'}).to_csv(TABLES_DIR / 'permutation_importance.csv', index=False)
    pd.DataFrame(confusion_rows).to_csv(TABLES_DIR / 'confusion_matrix.csv', index=False)
    discovery_report_row = {
        k: (', '.join(str(x) for x in v) if isinstance(v, list) else v)
        for k, v in DISCOVERY_REPORT.items()
    }
    pd.DataFrame([discovery_report_row]).to_csv(TABLES_DIR / 'dataset_discovery_report.csv', index=False)
metrics_per_fold_df

,fold_id,accuracy,precision,recall,specificity,sensitivity,balanced_accuracy,f1,roc_auc,mcc,tn,fp,fn,tp,n_pixels
0,1,0.888550,0.381988,0.644691,0.909669,0.644691,0.777180,0.479729,0.898356,0.440023,218438,21691,7389,13407,260925
1,2,0.898201,0.629841,0.768683,0.920883,0.768683,0.844783,0.692370,0.872293,0.636601,204472,17567,8995,29891,260925
2,3,0.868502,0.579313,0.182234,0.978742,0.182234,0.580488,0.277252,0.766018,0.272409,220033,4779,29532,6581,260925
3,4,0.821658,0.384715,0.938094,0.806652,0.938094,0.872373,0.545655,0.938206,0.528414,186448,44690,1844,27943,260925
4,5,0.755401,0.282829,0.919104,0.736922,0.919104,0.828013,0.432552,0.924860,0.421323,172778,61681,2141,24325,260925
5,6,0.946444,0.435173,0.868660,0.949901,0.868660,0.909281,0.579856,0.972330,0.592636,237308,12516,1458,9643,260925


In [ ]:
# Cell 17 — Summary
TOTAL_RUNTIME = time.perf_counter() - START_TIME
print('Reproducibility')
summary_line('Execution date', EXECUTION_DATE)
summary_line('Python version', sys.version.replace('\n', ' '))
summary_line('NumPy version', np.__version__)
summary_line('scikit-image version', __import__('skimage').__version__)
summary_line('scikit-learn version', __import__('sklearn').__version__)
summary_line('Random seed', RANDOM_SEED)
print('Runtime')
summary_line('Training time (sec)', round(TRAIN_END - TRAIN_START, 2))
summary_line('Evaluation time (sec)', round(EVAL_END - EVAL_START, 2))
summary_line('Total runtime (sec)', round(TOTAL_RUNTIME, 2))
print('Outputs')
summary_line('Results directory', RESULT_DIR.resolve())
summary_line('Figures directory', FIGURES_DIR.resolve())
summary_line('Tables directory', TABLES_DIR.resolve())
summary_line('Models directory', MODELS_DIR.resolve())
summary_line('Probability maps directory', PROBABILITY_MAPS_DIR.resolve())
summary_line('Cache directory', CACHE_DIR.resolve())
print('Dataset summary')
summary_line('Total discovered slices', len(SLICE_INFOS))
summary_line('CV slices', len(CV_SLICE_INFOS))
summary_line('Holdout slice', None if HOLDOUT_SLICE_INFO is None else HOLDOUT_SLICE_INFO.slice_number)
summary_line('Number of folds', len(FOLDS))
print('Primary metrics (median [min, max] across folds)')
for metric in PRIMARY_METRICS:
    row = summary_statistics_df.loc[metric]
    summary_line(metric, f"{row['median']:.3f} [{row['min']:.3f}, {row['max']:.3f}]")
print('Secondary metrics (median [min, max] across folds)')
for metric in SECONDARY_METRICS:
    row = summary_statistics_df.loc[metric]
    summary_line(metric, f"{row['median']:.3f} [{row['min']:.3f}, {row['max']:.3f}]")